# Deep Learning Text Comparison

## Data Preparation

In [1]:
import os
from pathlib import Path
import time

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "transaction_text_dataset.csv"
RANDOM_STATE = 42
MAX_TOKENS = 5_000
SEQUENCE_LENGTH = 80
EMBEDDING_DIMENSION = 32
BATCH_SIZE = 128
MAX_EPOCHS = 5
tf.keras.utils.set_random_seed(RANDOM_STATE)

In [2]:
data = pd.read_csv(DATA_PATH)
X = data["transaction_text"].astype(str)
y = data["is_fraud"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Dataset rows: {len(data):,}")
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
display(pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "train_percentage": y_train.value_counts(normalize=True).sort_index().mul(100),
    "test_count": y_test.value_counts().sort_index(),
    "test_percentage": y_test.value_counts(normalize=True).sort_index().mul(100),
}).rename(index={0: "Legitimate (0)", 1: "Fraudulent (1)"}))

Dataset rows: 30,000
Training rows: 24,000; test rows: 6,000


,train_count,train_percentage,test_count,test_percentage
is_fraud,,,,
Legitimate (0),22674,94.475,5668,94.466667
Fraudulent (1),1326,5.525,332,5.533333


In [3]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))

X_train_sequences = vectorizer(np.asarray(X_train)).numpy()
X_test_sequences = vectorizer(np.asarray(X_test)).numpy()
VOCABULARY_SIZE = len(vectorizer.get_vocabulary())
class_counts = y_train.value_counts().sort_index()
class_weights = {
    class_label: len(y_train) / (len(class_counts) * count)
    for class_label, count in class_counts.items()
}

print(f"Vocabulary size: {VOCABULARY_SIZE:,}")
print("Class weights:", class_weights)
print("Training sequence shape:", X_train_sequences.shape)

Vocabulary size: 5,000
Class weights: {0: 0.5292405398253506, 1: 9.049773755656108}
Training sequence shape: (24000, 80)


## Model Definitions

In [4]:
def compile_model(model: tf.keras.Model) -> tf.keras.Model:
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="roc_auc")],
    )
    return model

def embedding_input() -> tuple[tf.keras.layers.Input, tf.Tensor]:
    inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH,), dtype="int64")
    embeddings = tf.keras.layers.Embedding(
        input_dim=VOCABULARY_SIZE,
        output_dim=EMBEDDING_DIMENSION,
    )(inputs)
    return inputs, embeddings

def build_cnn() -> tf.keras.Model:
    inputs, embeddings = embedding_input()
    features = tf.keras.layers.Conv1D(32, 3, activation="relu")(embeddings)
    features = tf.keras.layers.GlobalMaxPooling1D()(features)
    features = tf.keras.layers.Dropout(0.2)(features)
    features = tf.keras.layers.Dense(16, activation="relu")(features)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(features)
    return compile_model(tf.keras.Model(inputs, outputs, name="cnn"))

def build_simple_rnn() -> tf.keras.Model:
    inputs, embeddings = embedding_input()
    features = tf.keras.layers.SimpleRNN(24, dropout=0.1)(embeddings)
    features = tf.keras.layers.Dense(16, activation="relu")(features)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(features)
    return compile_model(tf.keras.Model(inputs, outputs, name="simple_rnn"))

def build_lstm() -> tf.keras.Model:
    inputs, embeddings = embedding_input()
    features = tf.keras.layers.LSTM(24, dropout=0.1)(embeddings)
    features = tf.keras.layers.Dense(16, activation="relu")(features)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(features)
    return compile_model(tf.keras.Model(inputs, outputs, name="lstm"))

## Training and Evaluation

In [5]:
def train_and_evaluate(
    model_name: str,
    model_builder: callable,
) -> dict[str, float | str | int]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = model_builder()
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=1,
        restore_best_weights=True,
    )
    history = model.fit(
        X_train_sequences,
        y_train,
        validation_split=0.15,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=[early_stopping],
        verbose=2,
    )
    start_time = time.perf_counter()
    probabilities = model.predict(X_test_sequences, batch_size=BATCH_SIZE, verbose=0).ravel()
    inference_seconds = time.perf_counter() - start_time
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, probabilities),
        "Inference_Time_Seconds": inference_seconds,
        "Epochs_Trained": len(history.history["loss"]),
    }

results = []
for name, builder in [("CNN", build_cnn), ("Simple RNN", build_simple_rnn), ("LSTM", build_lstm)]:
    print(f"Training {name}...")
    results.append(train_and_evaluate(name, builder))

Training CNN...


Epoch 1/5


160/160 - 4s - 24ms/step - loss: 0.6809 - roc_auc: 0.5826 - val_loss: 0.6134 - val_roc_auc: 0.6745


Epoch 2/5


160/160 - 1s - 8ms/step - loss: 0.6444 - roc_auc: 0.6704 - val_loss: 0.6006 - val_roc_auc: 0.7108


...

160/160 - 6s - 36ms/step - loss: 0.6885 - roc_auc: 0.5344 - val_loss: 0.6325 - val_roc_auc: 0.6122


Epoch 2/5


160/160 - 3s - 21ms/step - loss: 0.6603 - roc_auc: 0.6284 - val_loss: 0.6252 - val_roc_auc: 0.6098


Epoch 3/5


160/160 - 4s - 26ms/step - loss: 0.6134 - roc_auc: 0.7176 - val_loss: 0.6379 - val_roc_auc: 0.5820


## Model Comparison

In [6]:
comparison = pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)
display(comparison.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "ROC_AUC": "{:.4f}",
    "Inference_Time_Seconds": "{:.4f}",
}))

best_f1_model = comparison.loc[comparison["F1"].idxmax()]
best_auc_model = comparison.loc[comparison["ROC_AUC"].idxmax()]
print(f"Best by F1: {best_f1_model['Model']} ({best_f1_model['F1']:.4f})")
print(f"Best by ROC-AUC: {best_auc_model['Model']} ({best_auc_model['ROC_AUC']:.4f})")
print("This comparison uses a fixed 0.5 threshold; no threshold tuning was performed.")

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Inference_Time_Seconds,Epochs_Trained
0,CNN,0.7243,0.1084,0.5512,0.1812,0.7094,0.2298,4
1,LSTM,0.8208,0.1190,0.3494,0.1775,0.6285,0.6771,3
2,Simple RNN,0.8493,0.0914,0.1928,0.1240,0.5852,0.7084,5


Best by F1: CNN (0.1812)
Best by ROC-AUC: CNN (0.7094)
This comparison uses a fixed 0.5 threshold; no threshold tuning was performed.
